In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langgraph.graph import MessagesState, StateGraph, END
from typing import List, Union
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_chroma import Chroma
import os
from langgraph.types import Command
from langchain.messages import HumanMessage, AIMessage
from langchain_core.tools import create_retriever_tool
from langgraph.prebuilt import ToolNode


d:\programming\LangChain impelentation\LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_dir=os.getcwd()
persistant_dir=os.path.join(current_dir,"database","Chroma_db")

In [3]:
embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

db=Chroma(
    persist_directory=persistant_dir,
    embedding_function=embedding,
    collection_metadata={"hnsw_space":"cosine"}
)

retriver=db.as_retriever(search_type='mmr',
                search_kwargs={
                    "k":3,
                    "fetch_k":10,
                    "lambda_multi":0.5
                })

retriver_tool=create_retriever_tool(
    retriever=retriver,
    name="retriver",
    description="Retrives the infromation from the documents"
)

tools=[retriver_tool]

In [4]:

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm_with_tool=llm.bind_tools(tools=tools)

bouncer_llm_prompt=ChatPromptTemplate([
    ("system","You are a decision maker AI assistant. Your duty is to identify if the users question is "
    "about one of the following topics. \n\n"
    "1. RAG\n"
    "2. The foundational architecture of Retrieval-Augmented Generation (RAG) systems\n'"
    "3. The core components of RAG pipelines\n"
    "4. The classification of user queries in RAG systems\n"
    "5. Training paradigms and knowledge injection strategies for RAG models\n"
    "6. Methodologies and datasets utilized for systematically evaluating RAG systems\n"
    "7. Advanced methodologies for enhancing traditional RAG frameworks\n"
    "8. Future opportunities, ethical challenges, and the development of trustworthy RAG\n"
    ""
    "If the users question is about one of this topic respond with 'Yes' otherwise repond 'No'."),
    MessagesPlaceholder(variable_name="messages")
])

answer_generator_llm_prompt=ChatPromptTemplate([
    ("system","You are an AI Assistnt. Your duty is to provide answer to the user query by "
    "calling tools to retrive information from the stored documents and by analyzing the chat "
    "history."),
    MessagesPlaceholder(variable_name="messages"),
    ("system","Do not use tools if the users question can be answered using the chat history. " \
    "If the information is not sufficient use tools to " \
    "retrive necessary infomation")
])

In [5]:
class AgentState(MessagesState):
    on_topic:str

In [6]:
answer_generator_chain=answer_generator_llm_prompt | llm_with_tool
bouncer_chain=bouncer_llm_prompt | llm

def bouncer_node(state:AgentState):
    decision=bouncer_chain.invoke(state)
    decision=decision.content.lower()
    if "yes" in decision:
        return Command(
            goto="answer_generator",
            update={"on_topic":"Yes"}
        )
    else:
        return Command(
            goto="off_topic_node",
            update={"on_topic":"No"}
        )
    
def generate_answer_node(state:AgentState):
    answer=answer_generator_chain.invoke(state)
    return {"messages":answer}

def router_node(state:AgentState):
    last_message=state["messages"][-1]
    if isinstance(last_message,AIMessage) and hasattr(last_message,"tool_calls") and len(last_message.tool_calls)>0:
        return "tools"
    else:
        return END
    
def off_topic_node(state:AgentState):
    return Command(
        goto=END,
        update={
            "messages":AIMessage(content="Sorry I cannot Answer that question!")
        }
    )
        
tool_node = ToolNode(tools)

In [7]:
graph=StateGraph(AgentState)
graph.add_node("bouncer",bouncer_node)
graph.add_node("answer_generator",generate_answer_node)

graph.add_node("off_topic_node",off_topic_node)
graph.add_node("tools",tool_node)

graph.add_edge("tools", "answer_generator")
graph.add_conditional_edges(
    "answer_generator",
    router_node,
)
graph.set_entry_point("bouncer")

app=graph.compile()

In [8]:
query="What are the future research directions for developing trustworthy, multi-lingual, and multimodal RAG architectures?"
result=app.invoke({
    "messages": [HumanMessage(content=query)],
    "documents":None,
    "on_topic":"Unknown"
})


In [9]:
result

{'messages': [HumanMessage(content='What are the future research directions for developing trustworthy, multi-lingual, and multimodal RAG architectures?', additional_kwargs={}, response_metadata={}, id='76e9418b-59e8-4cfb-9b4f-938cc7d6d1e9'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'retriver', 'arguments': '{"query": "Future research directions for developing trustworthy, multi-lingual, and multimodal RAG architectures"}'}, '__gemini_function_call_thought_signatures__': {'b56581b0-5ad2-4b62-b611-cde696ecb707': 'CswDARFNMg++XFegatPLN6NQy33AN4z8B+i2B1VYHQSmcBHkSz3qjDe4uelQN7NdLHAQuBiy1+51bSN/OVskNPf1wfvYyv+5hoAx2ekg2G0/mxAtqhxYLAwHHS6EShkc4QYCu5XR36IowBSZmPx7Xzzr4zH+brm6kkLe/s+uHOS84XArTA4ADs3ukcB4KCHgMpexIAxtvejvW+ngu88JRgJzPOtZTKGHHxrwtH9kbtKvg5pYjz0uTnH7k4DQ7VHPA6tcNcUmBrJVmIvSBsXXuwhXwH6ahYYa9TNU91xNtzBe6TyRBK2KgszahVxeW6T7K7ClQSnq7SBWaHw+tHefKF1Ggef2TGEDlNRAB+XkPW6N4qnEpyDZcxDj5nfafzgMBQO6ZBAoIISoMZwg6pEiiYcoAmjTRLuQvXhshaLRiwSUqaIJlqdF4vQVi63OUyV/wbLXmS

In [10]:
query="What is the major break through happened in robotics recently?"
result=app.invoke({
    "messages": [HumanMessage(content=query)],
    "on_topic":"Unknown"
})
result

{'messages': [HumanMessage(content='What is the major break through happened in robotics recently?', additional_kwargs={}, response_metadata={}, id='a6ea4c32-b4b3-4d82-8f6c-344a00f4d6e1'),
  AIMessage(content='Sorry I cannot Answer that question!', additional_kwargs={}, response_metadata={}, id='20e66a4e-1b43-40ab-903f-e3653c9a07ca', tool_calls=[], invalid_tool_calls=[])],
 'on_topic': 'No'}